# Asgmt4: Text Clustering and Classification

In this assignment, you will apply the unsupervised methods (KMeans, HDBSCAN, LDA, BERTopic) and supervised methods (Logistic Regression, Linear SVM, MLP, optional BERT fine-tuning) we covered in class on your own text dataset. 

The goal is not to run every method without thinking. Instead, you should **actively decide which methods make the most sense for your specific dataset and analysis question,** and be able to justify your choice.

Each question asks you to:
1. Apply the method (try multiple settings/models).
2. Inspect the output carefully (with examples from your data).
3. Reflect in writing on what worked, what failed, and which choice fits your task best.

## Load your data (for public sharing)

In [ ]:
import gdown
import pandas as pd

# Upload the data file to your Google Drive, and turn on the link sharing
# Replace 'YOUR_FILE_ID' with the actual file ID from your public Google Drive link e.g., "https://drive.google.com/file/d/YOUR_FILE_ID/view?usp=sharing"
file_id = None

# Name for the downloaded file in Colab
output_filename = None

gdown.download(id=file_id, output=output_filename, quiet=False)

print(f"File '{output_filename}' downloaded successfully!")

# Now you can load it with pandas
df_raw = pd.read_csv(output_filename)
display(df_raw.head())

In [ ]:
# Apply your text preprocessing pipeline from Asgmt2 (or a simplified version)
# Result should be stored in a clean text column you'll use for the rest of this notebook.
text_column = 'text'              # Replace with the actual name of your text column
clean_text_column = 'cleaned_text'

def preprocess(text):
    # Reuse / adapt from Asgmt2
    ...

df_raw[clean_text_column] = df_raw[text_column].apply(lambda x: preprocess(x))
display(df_raw[[text_column, clean_text_column]].head())

---
# Part A. Unsupervised: Clustering & Topic Modeling

In this part, you will discover structure in your corpus without using any label. Try multiple methods on the same data and reflect on what each one reveals (and hides).

## Q1: KMeans clustering on TF-IDF

- 1.1 Build a TF-IDF matrix of your cleaned text. 

- 1.2 Choose `k` using diagnostics
    - Fit `KMeans` for a range of `k` (e.g., `k = 2 .. 10`).
    - Plot the **elbow curve** (inertia) and **silhouette scores**.
    - Pick a `k` that is supported by the diagnostics *and* makes sense for your data. 
- 1.3 Fit final KMeans with your chosen `k`. For each cluster:
    - Show the cluster size and the **top 10 words** (from `kmeans.cluster_centers_`).
    - Print 2–3 **representative documents** (highest cluster strength).
    - If your data has a known categorical variable (sentiment, source, year, topic label), produce a `pd.crosstab(cluster, group)` to see whether clusters align with it.


In [ ]:
# 1.1 TF-IDF matrix

In [ ]:
# 1.2 Choose k (elbow + silhouette)

In [ ]:
# 1.3 Final KMeans: top words, representative docs, crosstab

## Q2: HDBSCAN clustering

HDBSCAN does not require you to specify `k`, handles non-spherical clusters, and flags noise points as `-1`.

- 2.1 Reduce dimensions before clustering
    - Project the TF-IDF matrix (or sentence embeddings — your choice; justify) into ~5–10 dimensions with UMAP (`n_components=10`, `metric='cosine'`).

- 2.2 Fit HDBSCAN on the reduced features. Compare:
    - Number of clusters found
    - Percentage of noise points (label `-1`)
    - Cluster size distribution

- 2.3 For your chosen setting, inspect the clusters
    - Top words per cluster (mean TF-IDF over cluster members, skip the noise label)
    - 2–3 representative documents per cluster
    - If your data has a known categorical variable (sentiment, source, year, topic label), produce a `pd.crosstab(cluster, group)` to see whether clusters align with it.


In [ ]:
# 2.1 UMAP dimensionality reduction

In [ ]:
# 2.2 HDBSCAN with multiple min_cluster_size values

In [ ]:
# 2.3 Inspect clusters: top words, representative docs, crosstab

## Q3: Topic modeling — LDA

Clustering forces each document into one group. Topic models treat documents as **mixtures** of topics.

- 3.1 Build input
    - Build a `CountVectorizer` matrix (LDA expects raw counts, not TF-IDF).

- 3.2 Fit LDA on the count matrix. 
    - Topic size distribution (dominant topic per document)
    - Topic strength (max topic probability per document)

- 3.3 For your chosen setting, inspect the topics
    - Top 10 words per topic (from `lda.components_`)
    - 2–3 representative documents per topic (highest topic strength)
    - If your data has a known categorical variable (sentiment, source, year, topic label), produce a `pd.crosstab(cluster, group)` to see whether clusters align with it.

Iterate 3.2 and 3.3 until you get `n_components` you like.


In [ ]:
# 3.1 CountVectorizer matrix

In [ ]:
# 3.2 Fit LDA (try multiple n_components)

In [ ]:
# 3.3 Inspect topics: top words, representative docs, crosstab

## Q4: Topic modeling — BERTopic

BERTopic chains sentence embeddings → UMAP → HDBSCAN → c-TF-IDF, so it captures semantic context that LDA's bag-of-words misses.

- 4.1 Build input
    - Encode your documents with a `SentenceTransformer`. Pick a model appropriate for your language (e.g., `all-MiniLM-L6-v2` for English, `paraphrase-multilingual-MiniLM-L12-v2` or `jhgan/ko-sroberta-multitask` for Korean).

- 4.2 Fit BERTopic on the embeddings.
    - Fit `BERTopic` with `calculate_probabilities=True`.
    - Topic size distribution (dominant topic per document; note the outlier `-1` group)
    - Topic strength (max topic probability per document)

- 4.3 For your chosen setting, inspect the topics
    - Top 10 words per topic (e.g., `topic_model.get_topic(i)` or `topic_model.visualize_barchart()`)
    - 2–3 representative documents per topic (highest topic strength; skip the `-1` outliers)
    - If your data has a known categorical variable (sentiment, source, year, topic label), produce a `pd.crosstab(cluster, group)` to see whether clusters align with it.

Iterate 4.2 and 4.3 (e.g., `nr_topics='auto'` vs. fixed values) until you get a topic structure you like.


In [ ]:
# 4.1 Encode documents with a SentenceTransformer

In [ ]:
# 4.2 Fit BERTopic (try multiple settings)

In [ ]:
# 4.3 Inspect topics: top words, representative docs, crosstab

# Part A reflections
Things to consider: 

- Which method produced the most **interpretable** groups / topics for *your* corpus? (Could you write a one-line description for each cluster/topic without strain?)
- How did the methods differ in **how many groups** they produced and in **how balanced** those groups were? Did one method collapse everything into a giant cluster, or split your data into too many tiny ones?
- How did **noise / outliers** behave? Were the documents that HDBSCAN / BERTopic flagged as `-1` truly junk, or were they meaningful content you don't want to throw away?
- Did the methods that use **semantic embeddings** (BERTopic, optionally HDBSCAN on embeddings) reveal structure that the bag-of-words methods (KMeans on TF-IDF, LDA) missed — or vice versa?
- Sanity check: if you have a known categorical variable, which method's groups **aligned with it most cleanly** in the crosstab? And — equally important — did any method find structure that *cuts across* your known label in an interesting way?


## Your pick for Part A
- Best method for my data:
- Why (2–4 sentences):


---
# Part B. Supervised: Text Classification

Pick a categorical column in your dataset to use as the **target label** (e.g., sentiment, source, topic, product category, year-bucket). Aim for at least 2–4 reasonably balanced classes. State below what label you chose and why.

### Target label
- Column name:
- Why this label is meaningful for your analysis:
- Class balance (paste `value_counts()` output):

In [ ]:
# Set up label encoding and train/test split
# - Use sklearn.preprocessing.LabelEncoder to encode the label as integers
# - Use train_test_split with stratify=y, random_state=42
# - Report a most-frequent-class baseline accuracy with DummyClassifier

## Q5: Feature engineering and model comparison

Build at least **three** of the following feature sets from your data. You may add more or replace one with something more appropriate to your dataset 

1. **Non-text metadata** — one-hot encode useful categorical columns (avoid columns that leak the label).
2. **Engineered text features** — scalar summaries like length, word count, sentiment polarity / subjectivity (TextBlob or a Korean equivalent).
3. **TF-IDF** — fit on the **training rows only**, then transform train and test (to avoid leakage).
4. **Top-N n-gram counts** — pick the top 25–50 bi/tri-grams from training and use their counts.
5. **Sentence embeddings** (optional, recommended for richer features) — mean-pooled SBERT vectors per document.

- 5.1 Build the feature sets. Print the shape `(n_train, n_features)` for each.

- 5.2 Train three classifiers on each feature set:
    - `LogisticRegression`
    - `LinearSVC`
    - `MLPClassifier` (one small hidden layer, e.g., 64 units)
    Wrap each in a `Pipeline` with `MaxAbsScaler` (works for sparse + dense). Report test accuracy.

- 5.3 Visualize results as a heatmap of `(feature set × model) → accuracy`.

In [ ]:
# 5.1 Build feature sets

In [ ]:
# 5.2 Train 3 classifiers on each feature set, collect accuracies

In [ ]:
# 5.3 Heatmap of accuracy by (feature set x model)

# Part B reflections

Things to consider:

- For your data, did the **features** matter more than the **model**, or the other way around? Where was the biggest jump in accuracy?
- Which feature set **surprised** you (positively or negatively)? Did combining feature sets help, hurt, or stay neutral?
- Looking at your best model's **classification report** and **confusion matrix**: which class is easiest / hardest to predict, and which two classes get confused most often? Does that confusion make sense given the content?
- For your analysis goal, do you need a **deployable, fast, interpretable** model (favoring Logistic Regression / Linear SVM on TF-IDF), or the **highest possible accuracy** even at higher cost (favoring BERT / large embeddings)?
- Sanity check: how much better is your best model than the most-frequent-class **baseline**? If the gap is small, the task may be too hard, the label too noisy, or your features too weak.



### Your pick for Part B
- Best (feature set, model) combo for my data:
- Why (2–4 sentences):